In [ ]:
#Vic

In [ ]:
# Read Fasta file
map = []
file = filename
open file
    for every other row in file
        map += row

In [ ]:
# Build Architecture

# Label states for consensus
```
gap_char = "-"
gap_count = 0
seq_len = length(map[0])
num_seqs = length(map)
col_states = []
m_state_counter = 0

#for each column
for i in range(seq_len):
    # for each sequence
    for j in range(num_seqs):
        if map[j][i] == gap_char:
            gap_count += 1
    # if it's an M, increment and add M state
    if gap_count / seq_len < .5:
        m_state_counter += 1
        col_states += f"M{m_state_counter}"
    # if it's an I, do not increment and add I state
    else:
        col_states += f"I{m_state_counter}"
    # reset gap count
    gap_count = 0


# Label sequences using state assignments
seq_states_temp = []
total_states = 0

#for each seq. first just give letters, numbers later
for row, sequence in enumerate(maps):
    # for each residue
    for column, residue in enumerate(sequence):
        if residue != gap_char:
            if col_states[column].startswith("M"):
                seq_states_temp[row][column] = "M"
            elif col_states[column].startswith("I"):
                seq_states_temp[row][column] = "I"
        elif residue == gap_char:
            if col_states[column].startswith("M"):
                seq_states_temp[row][column] = "D"
            elif col_states[column].startswith("I"):
                seq_states_temp[row][column] = "I"
            
#loop back thru to assign numbers
m_state_counter = 1
seq_states = []
    #for each sequence
    for row, sequence in enumerate(maps):
        # for each residue
        for column, residue in enumerate(sequence):
            # first col rules
            if column == 0:
                if seq_states_temp[row][column] == "M":                    
                    seq_states[row][column] = f"M{m_state_counter}"
                elif seq_states_temp[row][column] == "D"
                    seq_states[row][column] = f"D{m_state_counter}"
            else:
                if seq_states_temp[row][column] == "M":
                    m_state_counter += 1
                    seq_states[row][column] += f"I{m_state_counter}"
                elif seq_states_temp[row][column] == "I":
                    if seq_states[row][column -1] == "M"
                    
 ```

In [ ]:
# Score a new query
new_query = "APRGDSUYWE"
input_sequences = fasta_file_sequences

hmm = HMM()
hmm.profile(input_sequences)
posterior = hmm.forward_backward(new_query)
hmm.viterbi(new_query)

#update viterbi, foward, backward, fb

In [ ]:
# viterbi pseudocode
def viterbi():
    input: sequence, trans, emit, states
    return: optimal path
    matrix = {}
    traceback = {}

    # in log space
    matrix["Begin"] = log(1)
    # handle I0 start edge case
    matrix["I0"] = matrix["Begin"] + log(transition["Begin"]["I0"]) + log(emission["I0"][sequence[0]])
    traceback["I0"] = "Begin"
    
    all other states matrix[state] = -inf
    
    for each i, residue in enumerate(sequence):
        for each currstate in (Mi, Di, Ii-1):
            for each valid prevstate: # using transition rules
                if currstate is M or I:
                    score = matrix[prevstate] + log(transition[prevstate][currstate] + log(emit[currstate][residue]))
                elif currstate is D: #no emission
                    score = matrix[prevstate] + log(transition[prev][curr]
                if score > matrix[currstate]: #update when it's better, default -inf
                    update matrix[currstate] = score
                    update traceback[currstate] = prevstate

    path = trace back thru traceback dict End to Begin
    reverse path
    return path

In [ ]:
# forward, backward, forward backward pseudocode
def forward():
    input: sequence, trans, emit, states
    return: probmatrix, totalprob
    
    forward = {}
    forward["Begin"] = log(1)
    all other states forward[state] = -inf
    
    for each currstate in sequence
        if M or I:
            forward[currstate] = logaddexp over all prevstate:
                forward[prevstate] + log(trans[prevstate][currstate] + log(emit[currstate][residue]))
        elif D: # no emission
            forward[currstate] = logaddexp over all prevstate:
                forward[prevstate] + log(trans[prevstate][currstate]

    totalprob = logaddexp over all states:
        forward[currstate]
    return forward, totalprob
        
def backward():
    input: sequence, trans, emit, states
    return: probmatrix, totalprob
        
    backward = {}
    backward["End"] = log(1)
    all other states backward[state] = -inf
    
    for each currstate in sequence end to 1:
        if M or I:
            reverse[currstate] = logaddexp over all nextstate:
                reverse[nextstate] + log(trans[currstate][nextstate] + log(emit[currstate][residue]))
        elif D: # no emission
            reverse[currstate] = logaddexp over all nextstate:
                forward[currstate] + log(trans[currstate][nextstate]

    totalprob = logaddexp over all states:
        reverse[currstate]
    
    return reverse, totalprob

def forward_backward():
    input: seq, trans, emit, states
    return: posterior, path
        
    forward, totalf = forward()
    backward, totalb = backward()
    check if totalf and totalb sufficiently close
    total_prob = totalf+totalb/2

    posterior = {}

    for each i in len(sequence):
        for each state (M, I, D):
            posterior[state][i] = forward[state][i] + backward[state][i] - total_prob # log space

    # find best path
    path = []
    for each i in len(sequence):
        path.append argmax of M, I, D for posterior[state][i]

    return posterior, path